In [ ]:
"""FAST CS4048 Assignment 1, Version C scrapers.

Running this file reproduces both CSV files for roll number 23L_0905.
"""

from __future__ import annotations

import re
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from time import perf_counter

import pandas as pd
import requests
from bs4 import BeautifulSoup
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


ROLL_NO = "23L_0905"
STATIC_URL = "https://scrapeme.live/shop/"
DYNAMIC_URL = "https://web-scraping.dev/testimonials"
OUTPUT_DIR = Path(".")
TIMEOUT = (10, 45)


def make_session() -> requests.Session:
    """Create a connection-pooled session that retries temporary server failures."""
    retry = Retry(
        total=4,
        connect=4,
        read=4,
        backoff_factor=0.7,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset({"GET"}),
    )
    adapter = HTTPAdapter(max_retries=retry, pool_connections=10, pool_maxsize=10)
    session = requests.Session()
    session.headers.update({"User-Agent": "FAST-CS4048-student-scraper/1.0 (educational use)"})
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session


def text_or_empty(node) -> str:
    return node.get_text(" ", strip=True) if node else ""


def discover_static_catalog() -> tuple[list[dict], int, int]:
    """Follow the site's next links and return unique listing records."""
    session = make_session()
    url = STATIC_URL
    page_number = 1
    seen_pages: set[str] = set()
    products_by_url: dict[str, dict] = {}
    expected_total = None

    while url and url not in seen_pages:
        seen_pages.add(url)
        response = session.get(url, timeout=TIMEOUT)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")

        if expected_total is None:
            count_text = text_or_empty(soup.select_one("p.woocommerce-result-count"))
            match = re.search(r"of\s+([\d,]+)\s+results", count_text, re.I)
            if not match:
                raise ValueError(f"Could not parse result count from: {count_text!r}")
            expected_total = int(match.group(1).replace(",", ""))

        cards = soup.select("ul.products li.product")
        if not cards:
            raise RuntimeError(f"No products found on listing page {page_number}: {url}")
        for card in cards:
            link = card.select_one("a.woocommerce-LoopProduct-link")
            product_url = link.get("href", "").strip() if link else ""
            if product_url and product_url not in products_by_url:
                products_by_url[product_url] = {
                    "product_name": text_or_empty(card.select_one("h2.woocommerce-loop-product__title")),
                    "price": text_or_empty(card.select_one("span.price")),
                    "product_url": product_url,
                    "listing_page_number": page_number,
                }

        next_link = soup.select_one("nav.woocommerce-pagination a.next")
        url = next_link.get("href") if next_link else None
        page_number += 1

    session.close()
    return list(products_by_url.values()), int(expected_total), len(seen_pages)


_thread_local = threading.local()


def worker_session() -> requests.Session:
    # requests.Session is not guaranteed thread-safe, so each worker reuses its own pool.
    if not hasattr(_thread_local, "session"):
        _thread_local.session = make_session()
    return _thread_local.session


def scrape_product_detail(listing_record: dict) -> dict:
    response = worker_session().get(listing_record["product_url"], timeout=TIMEOUT)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    summary = soup.select_one("div.summary.entry-summary")
    if summary is None:
        raise ValueError("Product summary was not found")

    categories = [a.get_text(" ", strip=True) for a in summary.select("span.posted_in a")]
    tags = [a.get_text(" ", strip=True) for a in summary.select("span.tagged_as a")]
    return {
        **listing_record,
        "sku": text_or_empty(summary.select_one("span.sku")),
        "stock_availability": text_or_empty(summary.select_one("p.stock")),
        "categories": " | ".join(categories),
        "tags": " | ".join(tags),
        "short_description": text_or_empty(
            summary.select_one("div.woocommerce-product-details__short-description")
        ),
    }


def scrape_static_catalog(max_workers: int = 8) -> tuple[pd.DataFrame, dict]:
    started = perf_counter()
    listings, expected_total, pages = discover_static_catalog()
    records: list[dict] = []
    failures: list[dict] = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        future_to_item = {executor.submit(scrape_product_detail, item): item for item in listings}
        for future in as_completed(future_to_item):
            item = future_to_item[future]
            try:
                records.append(future.result())
            except Exception as exc:
                failures.append({"product_url": item["product_url"], "error": str(exc)})

    columns = [
        "product_name", "price", "sku", "stock_availability", "categories", "tags",
        "short_description", "product_url", "listing_page_number",
    ]
    df = pd.DataFrame(records, columns=columns)
    if not df.empty:
        df = (df.drop_duplicates(subset="product_url", keep="first")
                .sort_values(["listing_page_number", "product_name"], kind="stable")
                .reset_index(drop=True))
    output = OUTPUT_DIR / f"{ROLL_NO}_versionC_static_products.csv"
    df.to_csv(output, index=False, encoding="utf-8")
    required = ["product_name", "price", "sku", "stock_availability", "short_description", "product_url"]
    complete = int(df[required].fillna("").astype(str).apply(lambda c: c.str.strip().ne("")).all(axis=1).sum())
    stats = {
        "listing_pages_processed": pages,
        "site_reported_total": expected_total,
        "unique_listing_urls": len(listings),
        "records_written": len(df),
        "complete_required_records": complete,
        "failed_detail_requests": len(failures),
        "failures": failures,
        "elapsed_seconds": round(perf_counter() - started, 1),
        "count_check": "PASS" if len(df) == expected_total else "FAIL",
        "output_file": str(output),
    }
    print("STATIC COUNT CHECK:", stats["count_check"], f"({len(df)} unique / {expected_total} expected)")
    print(stats)
    return df, stats


def count_static_testimonials() -> int:
    """Confirm how many records plain requests can see without JavaScript."""
    with make_session() as session:
        response = session.get(DYNAMIC_URL, timeout=TIMEOUT)
        response.raise_for_status()
    return len(BeautifulSoup(response.text, "html.parser").select("div.testimonials div.testimonial"))


def scrape_dynamic_testimonials(headless: bool = True) -> tuple[pd.DataFrame, dict]:
    """Use Selenium, explicit waits, and two stable scrolls as the stop condition."""
    from selenium import webdriver
    from selenium.common.exceptions import TimeoutException
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support.ui import WebDriverWait

    started = perf_counter()
    raw_html_count = count_static_testimonials()
    options = webdriver.ChromeOptions()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--window-size=1280,900")
    options.add_argument("--disable-gpu")
    driver = webdriver.Chrome(options=options)
    records_by_key: dict[tuple[str, int], dict] = {}
    batch = 0
    stable_scrolls = 0
    scroll_attempts = 0

    def collect_visible(first_seen_batch: int) -> None:
        for element in driver.find_elements(By.CSS_SELECTOR, "div.testimonials div.testimonial"):
            text_nodes = element.find_elements(By.CSS_SELECTOR, "p.text")
            if not text_nodes:
                continue
            testimonial_text = text_nodes[0].text.strip()
            rating = len(element.find_elements(By.CSS_SELECTOR, "span.rating svg"))
            key = (testimonial_text, rating)
            records_by_key.setdefault(key, {
                "testimonial_text": testimonial_text,
                "rating": rating,
                "scroll_batch": first_seen_batch,
            })

    try:
        driver.get(DYNAMIC_URL)
        WebDriverWait(driver, 30).until(
            lambda d: len(d.find_elements(By.CSS_SELECTOR, "div.testimonials div.testimonial")) > 0
        )
        wait = WebDriverWait(driver, 8)
        collect_visible(0)

        while stable_scrolls < 2:
            previous_count = len(driver.find_elements(By.CSS_SELECTOR, "div.testimonials div.testimonial"))
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight)")
            scroll_attempts += 1
            try:
                wait.until(lambda d: len(d.find_elements(By.CSS_SELECTOR, "div.testimonials div.testimonial")) > previous_count)
                batch += 1
                stable_scrolls = 0
                collect_visible(batch)
            except TimeoutException:
                stable_scrolls += 1
                collect_visible(batch)
    finally:
        browser_count = len(records_by_key)
        driver.quit()

    columns = ["testimonial_text", "rating", "scroll_batch"]
    df = pd.DataFrame(records_by_key.values(), columns=columns).drop_duplicates(
        subset=["testimonial_text", "rating"], keep="first"
    )
    output = OUTPUT_DIR / f"{ROLL_NO}_versionC_dynamic_testimonials.csv"
    df.to_csv(output, index=False, encoding="utf-8")
    complete = int(df.fillna("").astype(str).apply(lambda c: c.str.strip().ne("")).all(axis=1).sum())
    stats = {
        "raw_html_testimonials": raw_html_count,
        "browser_testimonials": browser_count,
        "successful_growth_batches": batch,
        "scroll_attempts": scroll_attempts,
        "records_written": len(df),
        "complete_required_records": complete,
        "duplicates_after_deduplication": int(df.duplicated(["testimonial_text", "rating"]).sum()),
        "elapsed_seconds": round(perf_counter() - started, 1),
        "output_file": str(output),
    }
    print("DYNAMIC PREMISE CHECK:", f"requests={raw_html_count}, browser={browser_count}")
    print(stats)
    return df, stats


def playwright_bonus(headless: bool = True, growth_batches: int = 2) -> list[dict]:
    """Meaningful bonus demo: collect initial records and two dynamically loaded batches."""
    from playwright.sync_api import sync_playwright

    with sync_playwright() as playwright:
        browser = playwright.chromium.launch(headless=headless)
        page = browser.new_page(viewport={"width": 1280, "height": 900})
        page.goto(DYNAMIC_URL, wait_until="domcontentloaded")
        cards = page.locator("div.testimonials div.testimonial")
        cards.first.wait_for(state="visible")
        completed = 0
        while completed < growth_batches:
            before = cards.count()
            page.evaluate("window.scrollTo(0, document.body.scrollHeight)")
            page.wait_for_function(
                "previous => document.querySelectorAll('div.testimonials div.testimonial').length > previous",
                arg=before,
                timeout=12000,
            )
            completed += 1
        result = [
            {
                "testimonial_text": card.locator("p.text").inner_text().strip(),
                "rating": card.locator("span.rating svg").count(),
            }
            for card in cards.all()
        ]
        browser.close()
    print(f"Playwright collected {len(result)} testimonials after {completed} growth batches.")
    return result




In [ ]:
# IMPORTANT: set this to your real roll number before running.
ROLL_NO = '23L_0905'
static_df, static_stats = scrape_static_catalog(max_workers=8)
display(static_df.head(3))
display(static_df.tail(3))
assert static_df['product_url'].is_unique
static_stats

In [ ]:
dynamic_df, dynamic_stats = scrape_dynamic_testimonials(headless=True)
display(dynamic_df.head(3))
display(dynamic_df.tail(3))
display(dynamic_df.groupby('scroll_batch').size().rename('records'))
assert not dynamic_df.duplicated(['testimonial_text', 'rating']).any()
dynamic_stats

In [ ]:
# Optional bonus (uncomment after: playwright install chromium)
# playwright_records = playwright_bonus(headless=True, growth_batches=2)
# playwright_records[:3]

## Dataset publication

Public GitHub URL: https://github.com/faqsaeed/23L_0905

The published CSV files are the direct outputs produced by the scraper.